# Unit 7: 综合实战项目 - CIFAR-10图像分类

## 项目概述

本项目整合前6个单元的所有知识，完成一个完整的图像分类任务。我们将使用CIFAR-10数据集，从数据加载、模型构建、训练、评估到可视化，完整演练深度学习项目的全流程。

## 学习目标
- 整合所学知识完成完整的深度学习项目
- 掌握从数据处理到模型评估的全流程
- 学会超参数调优技巧
- 培养解决实际问题的能力

## CIFAR-10数据集
- 60,000张32x32彩色图像
- 10个类别，每类6,000张
- 训练集50,000张，测试集10,000张
- 类别：飞机、汽车、鸟、猫、鹿、狗、青蛙、马、船、卡车

## 参考资源
- [CIFAR-10数据集](https://www.cs.toronto.edu/~kriz/cifar.html)
- [PyTorch CIFAR-10教程](https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html)

## 7.1 导入必要的库

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.models import ResNet18_Weights
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import time
import os
from tqdm import tqdm

print("所有库已成功导入")
print(f"PyTorch版本: {torch.__version__}")
print(f"CUDA可用: {torch.cuda.is_available()}")

## 7.2 数据加载与预处理

In [ ]:
print("=" * 60)
print("7.2 数据加载与预处理")
print("=" * 60)

CIFAR10_CLASSES = ('plane', 'car', 'bird', 'cat', 'deer',
                  'dog', 'frog', 'horse', 'ship', 'truck')

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

print("正在下载/加载CIFAR-10数据集...")
train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=train_transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=test_transform
)

train_size = int(0.9 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_subset, val_subset = torch.utils.data.random_split(
    train_dataset, [train_size, val_size]
)

batch_size = 128
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, 
                         num_workers=2, pin_memory=True)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False,
                       num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                        num_workers=2, pin_memory=True)

print(f"训练集: {len(train_subset)} 样本")
print(f"验证集: {len(val_subset)} 样本")
print(f"测试集: {len(test_dataset)} 样本")
print(f"批次大小: {batch_size}")
print(f"训练批次: {len(train_loader)}")

## 7.3 数据可视化

In [ ]:
print("=" * 60)
print("7.3 数据可视化")
print("=" * 60)

def imshow(img, ax=None):
    img = img / 2 + 0.5
    npimg = img.numpy()
    if ax is None:
        plt.imshow(np.transpose(npimg, (1, 2, 0)))
    else:
        ax.imshow(np.transpose(npimg, (1, 2, 0)))

dataiter = iter(train_loader)
images, labels = next(dataiter)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
axes = axes.ravel()

for i in range(16):
    imshow(images[i], axes[i])
    axes[i].set_title(CIFAR10_CLASSES[labels[i]], fontsize=10)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

print("CIFAR-10数据集样本示例已显示")

## 7.4 定义CNN模型

In [ ]:
print("=" * 60)
print("7.4 定义CNN模型")
print("=" * 60)

class CIFAR10CNN(nn.Module):
    def __init__(self, num_classes=10):
        super(CIFAR10CNN, self).__init__()
        
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),
        )
        
        self.conv2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),
        )
        
        self.conv3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.classifier(x)
        return x

model = CIFAR10CNN(num_classes=10)
print(model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n总参数量: {total_params:,}")
print(f"可训练参数量: {trainable_params:,}")

dummy_input = torch.randn(2, 3, 32, 32)
output = model(dummy_input)
print(f"测试输入形状: {dummy_input.shape}")
print(f"测试输出形状: {output.shape}")

## 7.5 训练函数定义

In [ ]:
print("=" * 60)
print("7.5 训练函数定义")
print("=" * 60)

def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    progress_bar = tqdm(train_loader, desc='Training')
    
    for inputs, targets in progress_bar:
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        
        progress_bar.set_postfix({
            'loss': running_loss / total,
            'acc': 100.0 * correct / total
        })
    
    return running_loss / total, 100.0 * correct / total

def validate(model, val_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    return running_loss / total, 100.0 * correct / total

print("训练和验证函数已定义")

## 7.6 模型训练

In [ ]:
print("=" * 60)
print("7.6 模型训练")
print("=" * 60)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

model = CIFAR10CNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

num_epochs = 30
best_val_acc = 0.0
patience = 7
counter = 0

history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [],
    'lr': []
}

print(f"\n开始训练 {num_epochs} 个epoch...\n")
start_time = time.time()

for epoch in range(num_epochs):
    print(f"\nEpoch [{epoch+1}/{num_epochs}]")
    print("-" * 60)
    
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% | LR: {current_lr:.6f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        counter = 0
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
        }, 'best_cifar10_model.pth')
        print(f"✓ 保存最佳模型 (Val Acc: {val_acc:.2f}%)")
    else:
        counter += 1
        if counter >= patience:
            print(f"\n早停触发! 在epoch {epoch+1}停止训练")
            break

training_time = time.time() - start_time
print(f"\n训练完成!")
print(f"训练时间: {training_time:.2f}秒")
print(f"最佳验证准确率: {best_val_acc:.2f}%")

## 7.7 训练曲线可视化

In [ ]:
print("=" * 60)
print("7.7 训练曲线可视化")
print("=" * 60)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs_range = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_range, history['train_loss'], 'b-o', label='Train', linewidth=2, markersize=4)
axes[0].plot(epochs_range, history['val_loss'], 'r-o', label='Validation', linewidth=2, markersize=4)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Loss Curve', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['train_acc'], 'b-o', label='Train', linewidth=2, markersize=4)
axes[1].plot(epochs_range, history['val_acc'], 'r-o', label='Validation', linewidth=2, markersize=4)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('Accuracy Curve', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs_range, history['lr'], 'g-o', linewidth=2, markersize=4)
axes[2].set_xlabel('Epoch', fontsize=12)
axes[2].set_ylabel('Learning Rate', fontsize=12)
axes[2].set_title('Learning Rate Schedule', fontsize=14)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("训练曲线已绘制")

## 7.8 测试集评估

In [ ]:
print("=" * 60)
print("7.8 测试集评估")
print("=" * 60)

checkpoint = torch.load('best_cifar10_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])
print(f"加载最佳模型 (Epoch {checkpoint['epoch']}, Val Acc: {checkpoint['val_acc']:.2f}%)")

model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for inputs, targets in tqdm(test_loader, desc='Testing'):
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        _, predicted = outputs.max(1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)

test_acc = 100.0 * np.sum(all_preds == all_targets) / len(all_targets)
print(f"\n测试集准确率: {test_acc:.2f}%")

print(f"\n分类报告:")
print(classification_report(all_targets, all_preds, target_names=CIFAR10_CLASSES))

## 7.9 混淆矩阵分析

In [ ]:
print("=" * 60)
print("7.9 混淆矩阵分析")
print("=" * 60)

cm = confusion_matrix(all_targets, all_preds)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
           xticklabels=CIFAR10_CLASSES, yticklabels=CIFAR10_CLASSES,
           ax=axes[0])
axes[0].set_xlabel('Predicted Label', fontsize=12)
axes[0].set_ylabel('True Label', fontsize=12)
axes[0].set_title('Confusion Matrix', fontsize=14)

sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Greens',
           xticklabels=CIFAR10_CLASSES, yticklabels=CIFAR10_CLASSES,
           ax=axes[1])
axes[1].set_xlabel('Predicted Label', fontsize=12)
axes[1].set_ylabel('True Label', fontsize=12)
axes[1].set_title('Normalized Confusion Matrix', fontsize=14)

plt.tight_layout()
plt.show()

class_acc = cm_normalized.diagonal() * 100
print("\n各类别准确率:")
for i, cls in enumerate(CIFAR10_CLASSES):
    print(f"  {cls:8s}: {class_acc[i]:.2f}%")

best_class = CIFAR10_CLASSES[np.argmax(class_acc)]
worst_class = CIFAR10_CLASSES[np.argmin(class_acc)]
print(f"\n最佳类别: {best_class} ({np.max(class_acc):.2f}%)")
print(f"最差类别: {worst_class} ({np.min(class_acc):.2f}%)")

## 7.10 预测结果可视化

In [ ]:
print("=" * 60)
print("7.10 预测结果可视化")
print("=" * 60)

dataiter = iter(test_loader)
images, labels = next(dataiter)

model.eval()
with torch.no_grad():
    images = images.to(device)
    outputs = model(images)
    probabilities = F.softmax(outputs, dim=1)
    predictions = outputs.max(1)[1].cpu()

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
axes = axes.ravel()

for i in range(16):
    img = images[i].cpu() / 2 + 0.5
    npimg = img.numpy()
    axes[i].imshow(np.transpose(npimg, (1, 2, 0)))
    
    color = 'green' if predictions[i] == labels[i] else 'red'
    axes[i].set_title(f'True: {CIFAR10_CLASSES[labels[i]]}\nPred: {CIFAR10_CLASSES[predictions[i]]}',
                     color=color, fontsize=9)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

print("预测结果已可视化 (绿色=正确, 红色=错误)")

## 7.11 迁移学习对比实验(可选)

In [ ]:
print("=" * 60)
print("7.11 迁移学习对比实验")
print("=" * 60)

print("使用预训练ResNet-18进行迁移学习...")

resnet_model = models.resnet18(weights=ResNet18_Weights.DEFAULT)

for param in resnet_model.parameters():
    param.requires_grad = False

resnet_model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
resnet_model.maxpool = nn.Identity()

num_features = resnet_model.fc.in_features
resnet_model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(num_features, 256),
    nn.ReLU(),
    nn.Linear(256, 10)
)

resnet_model = resnet_model.to(device)

trainable_params = sum(p.numel() for p in resnet_model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in resnet_model.parameters())
print(f"ResNet-18迁移学习:")
print(f"  总参数量: {total_params:,}")
print(f"  可训练参数量: {trainable_params:,}")
print(f"  可训练比例: {trainable_params/total_params*100:.2f}%")

print("\n提示: 可以使用相同的训练流程训练此模型，与自定义CNN进行对比")

## 7.12 模型推理示例

In [ ]:
print("=" * 60)
print("7.12 模型推理示例")
print("=" * 60)

def predict_single_image(model, image_path=None, image_tensor=None, device='cpu'):
    model.eval()
    
    if image_tensor is None:
        from PIL import Image
        image = Image.open(image_path).convert('RGB')
        transform = transforms.Compose([
            transforms.Resize((32, 32)),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
        ])
        image_tensor = transform(image).unsqueeze(0)
    
    image_tensor = image_tensor.to(device)
    
    with torch.no_grad():
        output = model(image_tensor)
        probabilities = F.softmax(output, dim=1)
        predicted_class = output.max(1)[1].item()
        confidence = probabilities.max().item()
    
    return predicted_class, confidence, probabilities.cpu().numpy()[0]

test_image = torch.randn(1, 3, 32, 32)
pred_class, confidence, probs = predict_single_image(model, image_tensor=test_image, device=device)

print(f"预测类别: {CIFAR10_CLASSES[pred_class]}")
print(f"置信度: {confidence:.4f}")
print(f"\n所有类别的概率:")
for i, cls in enumerate(CIFAR10_CLASSES):
    print(f"  {cls:8s}: {probs[i]:.4f}")

## 7.13 超参数调优建议

In [ ]:
print("=" * 60)
print("7.13 超参数调优指南")
print("=" * 60)

print("关键超参数及其调优建议:\n")

print("1. 学习率(Learning Rate)")
print("   推荐范围: [1e-4, 1e-2]")
print("   策略: 使用学习率调度器(余弦退火、StepLR)")
print("   技巧: 学习率预热(warmup)\n")

print("2. 批次大小(Batch Size)")
print("   推荐范围: [32, 64, 128, 256]")
print("   策略: 较大batch需要较大学习率")
print("   技巧: 根据GPU内存调整\n")

print("3. 优化器(Optimizer)")
print("   推荐: Adam, AdamW, SGD+Momentum")
print("   Adam: 默认lr=0.001, 适合大多数任务")
print("   SGD: 需要手动调整学习率和动量\n")

print("4. 权重衰减(Weight Decay)")
print("   推荐范围: [1e-5, 1e-3]")
print("   作用: L2正则化，防止过拟合\n")

print("5. Dropout比率")
print("   推荐范围: [0.2, 0.5]")
print("   策略: 卷积层用0.25，全连接层用0.5\n")

print("6. 数据增强")
print("   策略: 根据数据集特点选择")
print("   CIFAR-10: RandomCrop, RandomFlip, RandomRotation")
print("   注意: 验证集不使用数据增强\n")

print("调参流程:")
print("  1. 先确定合理的batch size")
print("  2. 搜索最佳学习率")
print("  3. 调整正则化参数(Dropout, Weight Decay)")
print("  4. 优化数据增强策略")
print("  5. 尝试不同的优化器")

## 项目总结

恭喜！你已经完成了一个完整的CIFAR-10图像分类项目。

### 本项目涵盖的关键步骤：
1. **数据加载与预处理**：使用torchvision加载CIFAR-10数据集
2. **数据增强**：应用多种数据增强技术提高模型泛化能力
3. **模型构建**：设计并实现自定义CNN架构
4. **模型训练**：实现完整的训练循环，包括训练、验证、学习率调度
5. **模型评估**：使用多种指标评估模型性能
6. **结果可视化**：绘制训练曲线、混淆矩阵、预测结果
7. **迁移学习**：使用预训练模型进行对比实验
8. **模型推理**：实现单张图像的预测功能

### 进一步改进方向：
1. 尝试更深的网络架构(ResNet, DenseNet)
2. 使用更高级的数据增强技术(CutMix, MixUp)
3. 实现学习率预热(Learning Rate Warmup)
4. 使用标签平滑(Label Smoothing)
5. 集成多个模型的预测结果
6. 使用AutoAugment或RandAugment

### 学习成果检验：
通过完成本学习计划的全部7个单元，你应该已经掌握：
- CNN的基础理论和工作原理
- PyTorch框架的核心功能
- 模型构建、训练、评估的完整流程
- 迁移学习和预训练模型的使用
- 解决实际图像分类问题的能力

### 后续学习建议：
1. 阅读经典论文(ResNet, DenseNet, EfficientNet等)
2. 参加Kaggle竞赛实践
3. 学习目标检测、语义分割等高级任务
4. 探索Transformer在视觉领域的应用(ViT, Swin Transformer)
5. 关注PyTorch官方教程和最新研究进展

祝你在深度学习的道路上不断进步！